# Figure 3b: resistivity sections

Draws AEM resistivity sections for the three representative response classes.


In [ ]:
from pathlib import Path
import shutil
import subprocess

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (p / 'data').exists() and (p / 'notebooks' / 'resistivity_sections.jl').exists()
)
NOTEBOOK_DIR = ROOT / 'notebooks'
SCRIPT = NOTEBOOK_DIR / 'resistivity_sections.jl'
PROJECT_TOML = NOTEBOOK_DIR / 'Project.toml'

TIF_PATH = ROOT / 'data' / '1 resistivity' / 'aem_log10res_1km_masked.tif'
DEPTH_PATH = ROOT / 'data' / '1 resistivity' / 'aem_depth_levels_m.json'
LOOKUP_PATH = ROOT / 'data' / '2 well WTD' / 'active_cell_lookup.csv'
IDOMAIN_PATH = ROOT / 'data' / '1 resistivity' / 'idomain_1km_target.dat'
TOPO_PATH = ROOT / 'data' / '5 topography' / 'topo_1km.csv'
CLUSTER_LABELS_PATH = ROOT / 'outputs' / 'RECON_MAIN_2011_2023' / 'metrics' / 'clustering' / 'cluster_labels.csv'
FIG3_SELECTED_SITES_PATH = ROOT / 'outputs' / 'figures' / 'Fig3' / 'source_data' / 'fig3_selected_sites.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'figures' / 'Fig3'

TARGET_GRID_IDS = [45679, 83836, 84953]
TARGET_EXPECTED_CLASSES = ['Fast recovery', 'Slow recovery', 'Buffered']

HALF_WINDOW_KM = 20

VIEW_AZIMUTH_PI = [-0.3, 0.72, 0.72]
VIEW_ELEVATION_PI = [0.13, 0.13, 0.13]

def format_panel_values(values):
    if isinstance(values, (list, tuple)):
        return ','.join(str(v) for v in values)
    return str(values)

print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT.relative_to(ROOT))
print('JULIA PROJECT:', PROJECT_TOML.relative_to(ROOT))
print('TIF:', TIF_PATH.relative_to(ROOT))
print('DEPTHS:', DEPTH_PATH.relative_to(ROOT))
print('LOOKUP:', LOOKUP_PATH.relative_to(ROOT))
print('IDOMAIN:', IDOMAIN_PATH.relative_to(ROOT))
print('TOPO:', TOPO_PATH.relative_to(ROOT))
print('CLUSTER LABELS:', CLUSTER_LABELS_PATH.relative_to(ROOT))
print('SELECTED SITES:', FIG3_SELECTED_SITES_PATH.relative_to(ROOT))
print('Output dir:', OUTPUT_DIR.relative_to(ROOT))
print('Target grid ids:', TARGET_GRID_IDS)
print('View azimuth pi:', VIEW_AZIMUTH_PI)
print('View elevation pi:', VIEW_ELEVATION_PI)

In [ ]:
def run_cmd(args, *, cwd=ROOT, check=True):
    print('> ' + ' '.join(str(a) for a in args))
    proc = subprocess.run(
        [str(a) for a in args],
        cwd=cwd,
        text=True,
        encoding='utf-8',
        errors='replace',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc


def find_julia():
    candidates = []
    found = shutil.which('julia')
    if found:
        candidates.append(Path(found))
    juliaup = shutil.which('juliaup')
    if juliaup:
        try:
            proc = subprocess.run([juliaup, 'which', 'julia'], text=True, encoding='utf-8', errors='replace', stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            if proc.returncode == 0 and proc.stdout.strip():
                candidates.append(Path(proc.stdout.strip()))
        except Exception:
            pass
    common_roots = [
        Path.home() / 'AppData' / 'Local' / 'Programs',
        Path.home() / 'AppData' / 'Local' / 'Programs' / 'Julia',
        Path.home() / 'AppData' / 'Local' / 'Microsoft' / 'WinGet' / 'Packages',
        Path('C:/Program Files'),
    ]
    for root in common_roots:
        if root.exists():
            try:
                candidates.extend(root.rglob('julia.exe'))
            except Exception:
                pass
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return None

julia = find_julia()
if julia is None:
    print('Julia was not found on this machine.')
    print('Install Julia, then restart VS Code/terminal and rerun this notebook from this cell.')
    print('Recommended command: winget install --id Julialang.Julia -e --accept-source-agreements --accept-package-agreements')
else:
    print('Julia:', julia)
    run_cmd([julia, '--version'], check=False)

In [ ]:
INSTALL_JULIA_WITH_WINGET = False

if INSTALL_JULIA_WITH_WINGET and julia is None:
    winget = shutil.which('winget')
    if winget is None:
        raise RuntimeError('winget is not available. Please install Julia manually from https://julialang.org/downloads/')
    run_cmd([
        winget,
        'install',
        '--id', 'Julialang.Julia',
        '-e',
        '--accept-source-agreements',
        '--accept-package-agreements',
    ])
    print('Julia installation requested. Restart VS Code/terminal, then rerun from the Julia check cell.')
elif julia is None:
    print('Julia is still missing. Keep this cell unchanged if you prefer manual install.')
    print('Manual install command: winget install --id Julialang.Julia -e --accept-source-agreements --accept-package-agreements')
else:
    print('Julia is already available:', julia)

In [ ]:
INPUT_PATHS = [
    TIF_PATH,
    DEPTH_PATH,
    LOOKUP_PATH,
    IDOMAIN_PATH,
    TOPO_PATH,
    CLUSTER_LABELS_PATH,
    FIG3_SELECTED_SITES_PATH,
]
missing = [path for path in INPUT_PATHS if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required input files:\n' + '\n'.join(str(path) for path in missing))

for path in INPUT_PATHS:
    print('input ok:', path.relative_to(ROOT), f'({path.stat().st_size:,} bytes)')

labels = pd.read_csv(
    CLUSTER_LABELS_PATH,
    usecols=['grid_id', 'response_class', 'best_score', 'score_margin'],
)
target_classes = labels[labels['grid_id'].isin(TARGET_GRID_IDS)].copy()
if len(target_classes) != len(TARGET_GRID_IDS):
    missing_ids = sorted(set(TARGET_GRID_IDS) - set(target_classes['grid_id']))
    raise ValueError(f'Target grid ids are missing from cluster labels: {missing_ids}')

target_classes['grid_id'] = pd.Categorical(target_classes['grid_id'], categories=TARGET_GRID_IDS, ordered=True)
target_classes = target_classes.sort_values('grid_id').reset_index(drop=True)
target_classes['expected_class'] = TARGET_EXPECTED_CLASSES
if not (target_classes['response_class'].to_numpy() == target_classes['expected_class'].to_numpy()).all():
    raise ValueError('Fig3b target classes no longer match the current three-class clustering. Update TARGETS before plotting.')

print('\nCurrent Fig3b target classes:')
print(target_classes[['grid_id', 'response_class', 'best_score', 'score_margin']].to_string(index=False))

In [ ]:
if julia is None:
    raise RuntimeError(
        'Julia is not available yet. Install it first, restart VS Code/terminal, '
        'then rerun the Julia check cell. Recommended command: '
        'winget install --id Julialang.Julia -e --accept-source-agreements --accept-package-agreements'
    )

run_cmd([
    julia,
    f'--project={NOTEBOOK_DIR}',
    '-e',
    'using Pkg; Pkg.instantiate(); Pkg.precompile()',
])

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_cmd([
    julia,
    f'--project={NOTEBOOK_DIR}',
    SCRIPT,
    '--output',
    OUTPUT_DIR,
    '--output-prefix',
    'Fig3',
    '--half-window-km',
    str(HALF_WINDOW_KM),
    '--view-azimuth-pi',
    format_panel_values(VIEW_AZIMUTH_PI),
    '--view-elevation-pi',
    format_panel_values(VIEW_ELEVATION_PI),
    '--tif',
    TIF_PATH,
    '--depths',
    DEPTH_PATH,
    '--lookup',
    LOOKUP_PATH,
    '--idomain',
    IDOMAIN_PATH,
    '--topo',
    TOPO_PATH,
])

In [ ]:
from IPython.display import Image, display

for name in ['Fig3_resistivity_sections_2D.png', 'Fig3_resistivity_cutaway_3D.png']:
    path = OUTPUT_DIR / name
    print(path.relative_to(ROOT), 'exists=', path.exists())
    if path.exists():
        display(Image(filename=str(path)))

pdf_path = OUTPUT_DIR / 'Fig3_resistivity_sections_2D.pdf'
print(pdf_path.relative_to(ROOT), 'exists=', pdf_path.exists())